# 11. Evaluación oficial de los cuatro modelos

M0 y M1 se leen de las salidas guardadas por el cuaderno 08. M2 y M3 se leen de la inferencia oficial ya ejecutada con los pesos identificados en 10. No se entrena ni se vuelve a inferir ningún modelo.

La inferencia oficial de M2/M3 aplicó `preprocess_tweet(lang='es')`, `max_length=128`, truncamiento, solapamiento de 32 piezas, `return_overflowing_tokens=True`, relleno y promedio de probabilidades Softmax entre tramos.

## A. Verificar procedencia e integrar predicciones

- **Qué hace:** comprueba hashes, IDs y etiquetas antes de unir los cuatro resultados.
- **Para qué:** asegurar que cada cifra corresponda a la corrida oficial.
- **Por qué:** recalcular o mezclar corridas produciría cifras distintas a las documentadas.

In [ ]:
%pip install -q pandas openpyxl scikit-learn scipy

In [ ]:
from pathlib import Path
import hashlib
import itertools
import json
import math

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score, classification_report, cohen_kappa_score,
    confusion_matrix, f1_score,
)

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ = Path('/content/drive/MyDrive/TESIS_DATA_SCIENCE/COLAB')
else:
    RAIZ = Path.cwd()

PRED_08 = RAIZ / 'TABLAS/08_evaluacion/predicciones_prueba_v4.xlsx'
METRICAS_08 = RAIZ / 'TABLAS/08_evaluacion/metricas_v4.xlsx'
OFICIAL = RAIZ / 'historico/restauracion_pre_clasificacion_2026-10-25/tablas/12_evaluacion_m2_m3_protocolo_original'
PRED_M2_M3 = OFICIAL / 'predicciones_m2_m3_protocolo_original_v12.xlsx'
MET_M2_M3 = OFICIAL / 'metricas_globales_m2_m3_v12.xlsx'
META = OFICIAL / 'metadatos_evaluacion_v12.json'
MANIFIESTO_10 = RAIZ / 'TABLAS/10_entrenamiento_boliviano/manifest_corrida_oficial.json'
SALIDA = RAIZ / 'TABLAS/11_evaluacion_final'
SALIDA.mkdir(parents=True, exist_ok=True)

HASHES = {
    PRED_08: '7bcd3876a614eb3ff59bc84b28a3bf6a63253e5e92c8d8b335d83ebbaf931be7',
    METRICAS_08: '158c85e7d085f74e9390007bbc0a11e8595a8a122c9d3506629ffa5357a8eb34',
    PRED_M2_M3: '06b5aea8a882c8c4561a69d1577ddc05018003bb9ce4c9d266ffe8377c6d31a6',
}

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b''):
            h.update(bloque)
    return h.hexdigest()

for ruta, esperado in HASHES.items():
    if not ruta.is_file() or sha256(ruta) != esperado:
        raise ValueError(f'La entrada no coincide con la evidencia oficial: {ruta.name}')
if not MANIFIESTO_10.is_file():
    raise FileNotFoundError('Ejecuta primero el cuaderno 10.')

meta = json.loads(META.read_text(encoding='utf-8'))
protocolo = meta['protocolo_original_cuaderno_08']
controles_protocolo = {
    'max_length': protocolo['max_length'] == 128,
    'stride': protocolo['stride'] == 32,
    'segmentos': protocolo['return_overflowing_tokens'] is True,
    'agregacion': 'Softmax' in protocolo['agregacion'],
}
if not all(controles_protocolo.values()):
    raise ValueError('Los metadatos no describen el protocolo oficial esperado.')

pred08 = pd.read_excel(PRED_08, dtype=object, keep_default_na=False)
pred23 = pd.read_excel(PRED_M2_M3, dtype=object, keep_default_na=False)
met08 = pd.read_excel(METRICAS_08)
met23_guardadas = pd.read_excel(MET_M2_M3)
for tabla in (pred08, pred23):
    tabla['id'] = pd.to_numeric(tabla['id'], errors='raise').astype(int)
    tabla['etiqueta_polaridad'] = tabla['etiqueta_polaridad'].astype(str).str.strip().str.lower()
if pred08['id'].tolist() != pred23['id'].tolist():
    raise ValueError('Los IDs de M0/M1 y M2/M3 no coinciden.')
if pred08['etiqueta_polaridad'].tolist() != pred23['etiqueta_polaridad'].tolist():
    raise ValueError('Las etiquetas humanas no coinciden entre las fuentes.')

predicciones = pred08[['id', 'fuente', 'etiqueta_polaridad']].copy()
predicciones['M0_base_TASS'] = pred08['pred_base_TASS'].astype(str)
predicciones['M1_ajustado_SMPS'] = pred08['pred_ajustado_SMPS'].astype(str)
predicciones['M2_ajustado_boliviano'] = pred23['M2_ajustado_boliviano'].astype(str)
predicciones['M3_SMPS_boliviano'] = pred23['M3_SMPS_boliviano'].astype(str)
predicciones.to_excel(SALIDA / 'predicciones_cuatro_modelos.xlsx', index=False)
display(predicciones.head(10))

## B. Calcular métricas e incertidumbre

- **Qué hace:** calcula aciertos, intervalos, F1 macro, kappa y métricas por clase.
- **Para qué:** producir una tabla única a partir de las predicciones oficiales.
- **Por qué:** la clase positiva tiene cinco casos y requiere informar soporte e incertidumbre.

In [ ]:
CLASES = ['negativa', 'neutral', 'positiva']
MODELOS = ['M0_base_TASS', 'M1_ajustado_SMPS', 'M2_ajustado_boliviano', 'M3_SMPS_boliviano']
y = predicciones['etiqueta_polaridad'].to_numpy()
rng = np.random.default_rng(42)
indices_bootstrap = rng.integers(0, len(y), size=(5000, len(y)))

def intervalo_wilson(aciertos, n, z=1.96):
    p = aciertos / n
    denominador = 1 + z**2 / n
    centro = (p + z**2 / (2*n)) / denominador
    margen = z * math.sqrt((p*(1-p) + z**2/(4*n))/n) / denominador
    return centro - margen, centro + margen

globales, por_clase, matrices = [], [], {}
for nombre in MODELOS:
    yp = predicciones[nombre].to_numpy()
    reporte = classification_report(y, yp, labels=CLASES, output_dict=True, zero_division=0)
    aciertos = int((y == yp).sum())
    ic_acc = intervalo_wilson(aciertos, len(y))
    f1_boot = np.array([
        f1_score(y[idx], yp[idx], labels=CLASES, average='macro', zero_division=0)
        for idx in indices_bootstrap
    ])
    globales.append({
        'modelo': nombre, 'n': len(y), 'aciertos': aciertos,
        'accuracy': accuracy_score(y, yp),
        'ic95_accuracy_inferior': ic_acc[0], 'ic95_accuracy_superior': ic_acc[1],
        'p_binomial_mayor_0_75': float(binomtest(aciertos, len(y), 0.75, alternative='greater').pvalue),
        'precision_macro': reporte['macro avg']['precision'],
        'recall_macro': reporte['macro avg']['recall'],
        'f1_macro': reporte['macro avg']['f1-score'],
        'ic95_f1_inferior': float(np.quantile(f1_boot, 0.025)),
        'ic95_f1_superior': float(np.quantile(f1_boot, 0.975)),
        'kappa': cohen_kappa_score(y, yp, labels=CLASES),
    })
    for clase in CLASES:
        por_clase.append({
            'modelo': nombre, 'clase': clase,
            'precision': reporte[clase]['precision'], 'recall': reporte[clase]['recall'],
            'f1': reporte[clase]['f1-score'], 'soporte': int(reporte[clase]['support']),
        })
    matrices[nombre] = {
        'cantidad': confusion_matrix(y, yp, labels=CLASES),
        'porcentaje': 100 * confusion_matrix(y, yp, labels=CLASES, normalize='true'),
    }

metricas = pd.DataFrame(globales)
metricas_clase = pd.DataFrame(por_clase)

esperados = {'M0_base_TASS': 70, 'M1_ajustado_SMPS': 76,
             'M2_ajustado_boliviano': 63, 'M3_SMPS_boliviano': 64}
calculados = dict(zip(metricas['modelo'], metricas['aciertos']))
if calculados != esperados:
    raise ValueError(f'Los aciertos calculados no corresponden a la corrida oficial: {calculados}')

# Contraste contra las tablas fuente, no sustitución de los cálculos.
for nombre, patron in [('M0_base_TASS', 'base'), ('M1_ajustado_SMPS', 'ajustado')]:
    guardada = met08[met08.modelo.astype(str).str.contains(patron, case=False)].iloc[0]
    calculada = metricas[metricas.modelo == nombre].iloc[0]
    if abs(float(guardada.f1_macro) - float(calculada.f1_macro)) > 1e-10:
        raise ValueError(f'F1 no coincide para {nombre}.')
for nombre in ['M2_ajustado_boliviano', 'M3_SMPS_boliviano']:
    guardada = met23_guardadas[met23_guardadas.modelo == nombre].iloc[0]
    calculada = metricas[metricas.modelo == nombre].iloc[0]
    if abs(float(guardada.f1_macro) - float(calculada.f1_macro)) > 1e-10:
        raise ValueError(f'F1 no coincide para {nombre}.')

metricas.to_excel(SALIDA / 'metricas_globales.xlsx', index=False)
metricas_clase.to_excel(SALIDA / 'metricas_por_clase.xlsx', index=False)
with pd.ExcelWriter(SALIDA / 'matrices_confusion.xlsx', engine='openpyxl') as escritor:
    for nombre, contenido in matrices.items():
        corto = nombre.split('_')[0]
        for tipo, matriz in contenido.items():
            pd.DataFrame(matriz, index=CLASES, columns=CLASES).to_excel(
                escritor, sheet_name=f'{corto}_{tipo}'
            )
display(metricas.round(10).head(10))
display(metricas_clase.head(10).round(4))

## C. Comparaciones pareadas y manifiesto

- **Qué hace:** aplica McNemar-Holm y bootstrap pareado frente a M0.
- **Para qué:** analizar diferencias sobre los mismos 100 casos.
- **Por qué:** las observaciones no son muestras independientes entre modelos.

In [ ]:
comparaciones = []
for a, b in itertools.combinations(MODELOS, 2):
    acierta_a = predicciones[a].to_numpy() == y
    acierta_b = predicciones[b].to_numpy() == y
    n10 = int(np.sum(acierta_a & ~acierta_b))
    n01 = int(np.sum(~acierta_a & acierta_b))
    p = 1.0 if n10 + n01 == 0 else float(binomtest(n01, n10+n01, 0.5).pvalue)
    comparaciones.append({'modelo_a': a, 'modelo_b': b,
                          'a_acierta_b_falla': n10, 'a_falla_b_acierta': n01,
                          'p_mcnemar_exacto': p})
orden = np.argsort([x['p_mcnemar_exacto'] for x in comparaciones])
p_orden = np.array([comparaciones[i]['p_mcnemar_exacto'] for i in orden])
ajustadas = np.maximum.accumulate(np.minimum(1.0, p_orden * (len(comparaciones)-np.arange(len(comparaciones)))))
p_holm = np.empty(len(comparaciones)); p_holm[orden] = ajustadas
for fila, p in zip(comparaciones, p_holm):
    fila['p_holm'] = float(p); fila['significativa_0_05'] = bool(p < 0.05)
mcnemar = pd.DataFrame(comparaciones)
mcnemar.to_excel(SALIDA / 'mcnemar_holm.xlsx', index=False)

base = predicciones['M0_base_TASS'].to_numpy()
boot = []
for nombre in MODELOS[1:]:
    estimada = predicciones[nombre].to_numpy()
    diferencias = np.array([
        f1_score(y[idx], estimada[idx], labels=CLASES, average='macro', zero_division=0)
        - f1_score(y[idx], base[idx], labels=CLASES, average='macro', zero_division=0)
        for idx in indices_bootstrap
    ])
    boot.append({
        'modelo_comparado': nombre, 'referencia': 'M0_base_TASS',
        'diferencia_f1_observada': float(
            f1_score(y, estimada, labels=CLASES, average='macro', zero_division=0)
            - f1_score(y, base, labels=CLASES, average='macro', zero_division=0)),
        'ic95_diferencia_inferior': float(np.quantile(diferencias, 0.025)),
        'ic95_diferencia_superior': float(np.quantile(diferencias, 0.975)),
        'p_bootstrap_bilateral': min(1.0, 2*min(float((diferencias<=0).mean()), float((diferencias>=0).mean()))),
    })
bootstrap = pd.DataFrame(boot)
bootstrap.to_excel(SALIDA / 'bootstrap_pareado_vs_m0.xlsx', index=False)

manifiesto = {
    'estado': 'evaluacion_oficial_consolidada_sin_inferencia_nueva',
    'protocolo': protocolo,
    'hashes_entradas': {ruta.name: valor for ruta, valor in HASHES.items()},
    'resultados': {
        fila['modelo']: {
            'aciertos': int(fila['aciertos']), 'accuracy': float(fila['accuracy']),
            'f1_macro': float(fila['f1_macro']), 'kappa': float(fila['kappa'])
        } for fila in globales
    },
    'causa_discrepancia': (
        'Una ejecución posterior produjo pesos distintos por no determinismo del ajuste; '
        'se conserva como histórica y no sustituye la corrida que respalda la tesis.'
    ),
}
(SALIDA / 'manifest_evaluacion_oficial.json').write_text(
    json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8'
)
display(mcnemar.head(10))
display(bootstrap.head(10))